# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Freshness and performance

The paper reports that content freshness is associated with differences in search performance.

My methodology question is: how is the outcome label defined for this finding, and does the validation design separate the information used to define freshness from the later performance outcome?

The comparison is useful as an observed relationship, but I would be careful about interpreting it as proof that refreshing a page directly causes the observed performance change. A stronger causal claim would require a design that controls for other differences between refreshed and non-refreshed pages.

### Finding 2 — Refreshed vs. stale content

The paper reports a measured difference in impressions between refreshed and stale content.

My methodology question is: does the validation design support a causal interpretation of the difference, or is it primarily an observational comparison?

Pages that are refreshed may already differ from stale pages in age, prior performance, content quality, or other factors. Therefore, I would treat the reported difference as measured evidence of an association unless the study design can rule out these alternative explanations.

These questions are intended as constructive methodology checks. The findings are useful, but the strength of the claim should match the evidence and validation design.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# LOAD WEEK-5 DATASET
# ==========================================

df = pd.read_csv(
    "https://raw.githubusercontent.com/shivanilokh/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# ==========================================
# CREATE TARGET
# ==========================================

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

target = "is_declining_label"

# ==========================================
# REMOVE LEAKAGE / ID COLUMNS
# ==========================================

exclude_cols = {
    target,
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
}

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
]

print("\nFeatures used by the model:")
print(feature_cols)

# ==========================================
# MODEL FUNCTION
# ==========================================

def build_model(X_train, y_train):

    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", rf)
        ]
    )

    model.fit(X_train, y_train)

    return model


# ==========================================
# PRECISION@50 FUNCTION
# ==========================================

def precision_at_50(model, test_df):

    X_test = test_df[feature_cols]

    probabilities = model.predict_proba(X_test)[:, 1]

    results = test_df[
        ["content_id", "client_id", target]
    ].copy()

    results["decline_probability"] = probabilities

    results = results.sort_values(
        "decline_probability",
        ascending=False
    )

    top_50 = results.head(50)

    precision = top_50[target].mean()

    return precision, results


# ==========================================
# BEFORE: RANDOM ROW SPLIT
# ==========================================

train_random, test_random = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[target]
)

random_model = build_model(
    train_random[feature_cols],
    train_random[target]
)

random_precision, random_results = precision_at_50(
    random_model,
    test_random
)


# ==========================================
# AFTER: CLIENT-GROUPED SPLIT
# ==========================================

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_grouped = df[
    df["client_id"].isin(train_clients)
].copy()

test_grouped = df[
    df["client_id"].isin(test_clients)
].copy()

grouped_model = build_model(
    train_grouped[feature_cols],
    train_grouped[target]
)

grouped_precision, grouped_results = precision_at_50(
    grouped_model,
    test_grouped
)


# ==========================================
# CHECK CLIENT OVERLAP
# ==========================================

client_overlap = len(
    set(train_grouped["client_id"])
    &
    set(test_grouped["client_id"])
)


# ==========================================
# BEFORE vs AFTER COMPARISON
# ==========================================

comparison = pd.DataFrame({
    "Validation Design": [
        "Random row split",
        "Client-grouped split"
    ],
    "Precision@50": [
        random_precision,
        grouped_precision
    ]
})


print("\n==========================================")
print("VALIDATION RESULTS")
print("==========================================")

print(
    "Random split Precision@50:",
    round(random_precision, 3)
)

print(
    "Client-grouped Precision@50:",
    round(grouped_precision, 3)
)

print(
    "Client overlap in grouped split:",
    client_overlap
)

print("\nBefore vs After:")
display(comparison)

Dataset loaded successfully!
Dataset shape: (30000, 44)

Features used by the model:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

VALIDATION RESULTS
Random split Precision@50: 1.0
Client-grouped Precision@50: 1.0
Client overlap in grouped split: 0

Before vs After:


,Validation Design,Precision@50
0,Random row split,1.0
1,Client-grouped split,1.0


### Before vs After

The Week-5 model was evaluated using two validation designs.

The random row split produced a measured Precision@50 of 1.00.

The client-grouped split also produced a measured Precision@50 of 1.00.

The grouped split had 0 client overlap between the training and test sets, making it a more honest test of generalization to unseen clients.

Because both validation approaches produced the same measured Precision@50 in this run, the grouped split did not reduce the measured top-50 precision on this dataset.

I treat this result as observed and directional rather than as proof that the model will perform the same way on future or unseen data.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# Leakage audit

label_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

identifier_columns = [
    "content_id",
    "client_id"
]

print("Target:", target)

print("\nExcluded label-derived / identifier columns:")

for col in label_columns + identifier_columns:
    print("-", col)

print("\nFeatures actually used by the model:")
print(feature_cols)

leakage_present = any(
    col in feature_cols
    for col in label_columns + identifier_columns
)

print(
    "\nLeakage columns present in final features:",
    leakage_present
)

Target: is_declining_label

Excluded label-derived / identifier columns:
- trend_direction
- trend_pct
- is_declining_label
- content_id
- client_id

Features actually used by the model:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Leakage columns present in final features: False


### Leakage audit result

I checked the final feature list for columns that directly define the target or identify the client or content item.

The target is created from `trend_direction`, so `trend_direction` and `trend_pct` were excluded from the model features.

`client_id` and `content_id` were also excluded from model features because they are identifiers rather than predictive signals.

The leakage check confirmed that these excluded columns were not present in the final feature set.

This reduces the risk of target leakage and client memorization in the model inputs.

In [9]:
# Real failure examples from the client-grouped test set

grouped_results["predicted_label"] = (
    grouped_results["decline_probability"] >= 0.5
).astype(int)

false_positives = grouped_results[
    (grouped_results["predicted_label"] == 1)
    & (grouped_results[target] == 0)
]

false_negatives = grouped_results[
    (grouped_results["predicted_label"] == 0)
    & (grouped_results[target] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")

display(
    false_positives[
        [
            "content_id",
            "client_id",
            "decline_probability",
            target
        ]
    ].head(5)
)

print("\nExample false negatives:")

display(
    false_negatives[
        [
            "content_id",
            "client_id",
            "decline_probability",
            target
        ]
    ].head(5)
)

False positives: 369
False negatives: 26

Example false positives:


,content_id,client_id,decline_probability,is_declining_label
2357,content_8f1409b2674e,client_8527a891e2,0.893333,0
12332,content_4d9f36001f06,client_8527a891e2,0.886667,0
20736,content_41baf0722ad9,client_8527a891e2,0.870000,0
1439,content_5585a0e7089c,client_8527a891e2,0.846667,0
28582,content_f49660e074e9,client_8527a891e2,0.810000,0



Example false negatives:


,content_id,client_id,decline_probability,is_declining_label
29152,content_743f469dddea,client_9400f1b21c,0.496667,1
18171,content_24796d98b025,client_9f14025af0,0.493333,1
26237,content_506289517701,client_a88a7902cb,0.493333,1
6713,content_aaa7641512c3,client_9400f1b21c,0.490000,1
2002,content_a6f36e44f79b,client_a88a7902cb,0.486667,1


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Random Forest model predicts which pages will decline and can identify the pages that should be refreshed.

### Rewritten claim

The Random Forest model measured Precision@50 on the evaluated dataset and can be used as decision-support for prioritizing pages for human review.

The result is observed and directional. It does not prove that the model will predict future page performance for every client, and it does not establish that refreshing a page will cause traffic recovery.

The model should therefore be treated as a ranking and review-support tool rather than an automatic refresh decision system.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.